In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import StaleElementReferenceException, ElementClickInterceptedException, NoSuchElementException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import random
import csv

# Konfiguracja Selenium
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Uruchamianie WebDrivera
service = Service(ChromeDriverManager().install())


# Ścieżka do pliku CSV
CSV_PATH = r"C:\Users\Laptop\Desktop\otodom_oferty.csv"

# Adres strony głównej
BASE_URL = "https://www.otodom.pl/pl/wyniki/sprzedaz/mieszkanie/cala-polska"

# Zamknięcie pop-upu cookies
def close_cookies_popup(driver):
    try:
        cookie_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button#onetrust-accept-btn-handler"))
        )
        cookie_button.click()
        print("Zamknięto okno cookies.")
        time.sleep(2)
    except TimeoutException:
        print("Brak okna cookies do zamknięcia.")

def expand_building_details(driver):
    try:

        header = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, '//header[p[contains(text(), "Budynek i materiały")]]'))
        )

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", header)
        time.sleep(1)
        header.click()
        time.sleep(2)  

        details_section = driver.find_elements(By.XPATH, '//div[contains(@class, "css-1ftuvmu") and not(@hidden)]')
        if details_section:
            print("Sekcja 'Budynek i materiały' rozwinięta!")
        else:
            print("Selenium nadal nie widzi sekcji!")

    except Exception as e:
        print(f"Błąd: {e}")

def get_value_after_label(label, driver):
    try:
        element = driver.find_element(By.XPATH, f'//p[text()="{label}"]/following-sibling::p')
        return element.text.strip()
    except NoSuchElementException:
        return "Brak danych"

def get_value_from_details(label, driver):
    try:
        time.sleep(2) 

        elements = driver.find_elements(By.XPATH, f'//p[contains(text(), "{label}")]/following-sibling::p')
        if elements:
            return elements[0].text.strip()
        else:
            print(f"Selenium nie znalazło '{label}'")
            return "Brak danych"
    except Exception as e:
        print(f"Błąd przy pobieraniu '{label}': {e}")
        return "Brak danych"


def get_value_from_buttons(index, driver):
    try:
        buttons = driver.find_elements(By.CSS_SELECTOR, 'button.eezlw8k1.css-1nk40gi')
        if len(buttons) > index:
            return buttons[index].find_element(By.CSS_SELECTOR, 'div.css-1ftqasz').text.strip()
        return "Brak danych"
    except NoSuchElementException:
        return "Brak danych"

# Pobieranie linków do ofert
def get_offer_links(driver, max_links=2000):
    driver.get(BASE_URL)
    wait = WebDriverWait(driver, 15)  

    close_cookies_popup(driver)

    all_links = set()
    previous_links = set() 

    while len(all_links) < max_links:
        try:
            # Pobranie ofert
            offers = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'a[href^="/pl/oferta/"]')))
            new_links = {offer.get_attribute('href') for offer in offers if offer.get_attribute('href')}
            
            # Sprawdzenie, czy strona rzeczywiście się zmieniła
            if new_links == previous_links:
                print("Strona nie zmieniła ofert. Próbuję ponownie...")
                time.sleep(3)
                continue  # Powtórz próbę pobrania

            previous_links = new_links  # Aktualizacja poprzednich linków
            all_links.update(new_links)
            print(f"🔹 Pobrano {len(new_links)} nowych ofert, łącznie: {len(all_links)}")

            # Sprawdzenie limitu ofert
            if len(all_links) >= max_links:
                print("Osiągnięto limit ofert, kończenie zbierania linków.")
                break

            # Przewinięcie do przycisku "Następna strona"
            try:
                next_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//li[@title="Go to next Page" and @aria-disabled="false"]')))
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_button)
                time.sleep(random.uniform(1, 2))

                # Ponawianie prób kliknięcia
                attempts = 0
                while attempts < 3:
                    try:
                        next_button.click()
                        time.sleep(random.uniform(4, 6))  # Dłuższe czekanie na załadowanie
                        break
                    except ElementClickInterceptedException:
                        print("Kliknięcie nieudane, ponawiam próbę...")
                        driver.execute_script("arguments[0].click();", next_button)
                        time.sleep(1)
                        attempts += 1

            except (NoSuchElementException, TimeoutException):
                print("Nie znaleziono przycisku paginacji. Koniec zbierania linków.")
                break

        except StaleElementReferenceException:
            print("Strona została zaktualizowana, ponawiam próbę...")
            continue  

    return list(all_links)[:max_links]   

# Pobieranie szczegółów oferty
def get_offer_details(offer_url, driver):
    driver.get(offer_url)
    time.sleep(random.uniform(2, 4))

    data = {"URL": offer_url}

    def get_text(selector, driver):
        try:
            return driver.find_element(By.CSS_SELECTOR, selector).text.strip()
        except:
            return "Brak danych"

    data["Cena"] = get_text('strong[data-cy="adPageHeaderPrice"]', driver)
    data["Cena za m²"] = get_text('div[aria-label="Cena za metr kwadratowy"]', driver)
    data["Adres"] = get_text('a.css-1jjm9oe', driver)
    data["Powierzchnia (m²)"] = get_value_from_buttons(0, driver)
    data["Liczba pokoi"] = get_value_from_buttons(1, driver)
    data["Ogrzewanie"] = get_value_after_label("Ogrzewanie", driver)
    data["Rynek"] = get_value_after_label("Rynek", driver)
    data["Typ ogłoszeniodawcy"] = get_value_after_label("Typ ogłoszeniodawcy", driver)

    expand_building_details(driver)
    time.sleep(15)
    data["Rok budowy"] = get_value_from_details("Rok budowy", driver)
    time.sleep(5)
    data["Rodzaj zabudowy"] = get_value_from_details("Rodzaj zabudowy", driver)
    time.sleep(5)
    data["Okna"] = get_value_from_details("Okna", driver)
    time.sleep(5)
    data["Materiał budynku"] = get_value_from_details("Materiał budynku", driver)

    print("\n Oferta pobrana:")
    for key, value in data.items():
        print(f"   {key}: {value}")

    return data

def scrape_otodom(offer_links, driver):
    all_data = []

    for idx, link in enumerate(offer_links, 1):
        print(f"\n Pobieranie oferty numer {idx}: {link}")
        details = get_offer_details(link, driver)
        all_data.append(details)
        time.sleep(random.uniform(2, 5))

    save_to_csv(all_data)

def save_to_csv(data):
    keys = data[0].keys() if data else []
    
    with open(CSV_PATH, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=keys)
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n Dane zapisane do pliku: {CSV_PATH}")


In [ ]:
driver = webdriver.Chrome(service=service, options=options)
offer_links = get_offer_links(driver)

Zamknięto okno cookies.
🔹 Pobrano 39 nowych ofert, łącznie: 39
🔹 Pobrano 39 nowych ofert, łącznie: 69
🔹 Pobrano 36 nowych ofert, łącznie: 105
🔹 Pobrano 39 nowych ofert, łącznie: 144
🔹 Pobrano 39 nowych ofert, łącznie: 182
🔹 Pobrano 38 nowych ofert, łącznie: 220
🔹 Pobrano 39 nowych ofert, łącznie: 258
🔹 Pobrano 38 nowych ofert, łącznie: 296
🔹 Pobrano 39 nowych ofert, łącznie: 334
🔹 Pobrano 39 nowych ofert, łącznie: 369
🔹 Pobrano 39 nowych ofert, łącznie: 406
🔹 Pobrano 38 nowych ofert, łącznie: 441
🔹 Pobrano 39 nowych ofert, łącznie: 479
🔹 Pobrano 38 nowych ofert, łącznie: 515
🔹 Pobrano 39 nowych ofert, łącznie: 553
🔹 Pobrano 39 nowych ofert, łącznie: 590
🔹 Pobrano 39 nowych ofert, łącznie: 627
🔹 Pobrano 39 nowych ofert, łącznie: 660
🔹 Pobrano 38 nowych ofert, łącznie: 696
🔹 Pobrano 39 nowych ofert, łącznie: 733
🔹 Pobrano 39 nowych ofert, łącznie: 771
🔹 Pobrano 39 nowych ofert, łącznie: 808
🔹 Pobrano 39 nowych ofert, łącznie: 843
🔹 Pobrano 39 nowych ofert, łącznie: 880
🔹 Pobrano 39 nowyc

In [13]:
scrape_otodom(offer_links, driver)
driver.quit() 


 Pobieranie oferty numer 1: https://www.otodom.pl/pl/oferta/funkcjonalne-mieszkanie-w-zdrowym-klimacie-ID4weew
Selenium nadal nie widzi sekcji!

 Oferta pobrana:
   URL: https://www.otodom.pl/pl/oferta/funkcjonalne-mieszkanie-w-zdrowym-klimacie-ID4weew
   Cena: 564 052,03 zł
   Cena za m²: 14 797 zł/m²
   Adres: ul. Szkolna, Międzywodzie, Dziwnów, kamieński, zachodniopomorskie
   Powierzchnia (m²): 38.12m²
   Liczba pokoi: 2 pokoje
   Ogrzewanie: kotłownia
   Rynek: pierwotny
   Typ ogłoszeniodawcy: biuro nieruchomości
   Rok budowy: 2026
   Rodzaj zabudowy: blok
   Okna: plastikowe
   Materiał budynku: silikat

 Pobieranie oferty numer 2: https://www.otodom.pl/pl/oferta/3-pokojowy-apartament-z-garazem-blisko-arkadii-ID4uYO9
Selenium nadal nie widzi sekcji!


KeyboardInterrupt: 